# Reading NWB files

This notebook is used to inspect the contents of a Neurodata Without Borders (NWB) file. The goal is to become familiar with the structure of the NWB file and to check whether the expected experimental data and metadata are present.

## 0. Import libraries and functions

In [1]:
from pynwb import read_nwb, NWBHDF5IO
import matplotlib.pyplot as plt
import numpy as np

from mrondes_files.helper_functions import select_file

## 1. Select file you want to inspect

In [2]:
filepath = select_file("select nwb file you want to read", filetypes=(("NWB files", "*.nwb"), ("All files", "*.*")))

## 2. Read file 

In [13]:
with NWBHDF5IO(filepath, "r", load_namespaces=True) as io:
    nwb = io.read()

    # top-level keys
    print("Acquisition:", list(nwb.acquisition.keys()))
    print("Processing:", list(nwb.processing.keys()))
    print("Devices:", list(nwb.devices.keys()))
    print("Electrode groups:", list(nwb.electrode_groups.keys()))

    # subject metadata
    print("\nSubject:")
    print("  ID:", nwb.subject.subject_id)
    print("  Sex:", nwb.subject.sex)
    print("  Genotype:", nwb.subject.genotype)
    print("  Weight:", nwb.subject.weight)
    print(" Description", nwb.subject.description)

    # electrodes info 
    df = nwb.electrodes.to_dataframe()
    print("\nElectrodes:")
    print(df) 

Acquisition: ['TTL_1', 'filtered_EEG', 'raw_EEG']
Processing: []
Devices: ['1023']
Electrode groups: ['EEG 10', 'EEG 11', 'EEG 12', 'EEG 13', 'EEG 3', 'EEG 4', 'EEG 6', 'EEG 7', 'EEG 8', 'EEG 9']

Subject:
  ID: b2c3.3
  Sex: f
  Genotype: 3TG
  Weight: None
 Description b2c3.3

Electrodes:
   location                                              group        label  \
id                                                                            
0      AI_L  EEG 3 pynwb.ecephys.ElectrodeGroup at 0x131741...   depth_AI_L   
1      S1_L  EEG 4 pynwb.ecephys.ElectrodeGroup at 0x131741...   skull_S1_L   
2      A1_L  EEG 7 pynwb.ecephys.ElectrodeGroup at 0x131741...   skull_A1_L   
3      A1_R  EEG 9 pynwb.ecephys.ElectrodeGroup at 0x131741...   skull_A1_R   
4     ACA_R  EEG 12 pynwb.ecephys.ElectrodeGroup at 0x13174...  depth_ACA_R   
5      AI_R  EEG 10 pynwb.ecephys.ElectrodeGroup at 0x13174...   depth_AI_R   
6     PMC_R  EEG 11 pynwb.ecephys.ElectrodeGroup at 0x13174...  depth_PMC_R 

In [25]:
with NWBHDF5IO(filepath, "r", load_namespaces=True) as io:
    nwb = io.read()
    raw = nwb.acquisition["raw_EEG"]
    filt = nwb.acquisition["filtered_EEG"]

    print(type(raw))
    print("name:", raw.name)
    print("description:", raw.description)
    print("unit:", raw.unit)
    print("data shape:", raw.data.shape)
    print("rate:", getattr(raw, "rate", None))
    print("starting_time:", getattr(raw, "starting_time", None)) 

<class 'pynwb.ecephys.ElectricalSeries'>
name: raw_EEG
description: no description
unit: volts
data shape: (678125, 10)
rate: 1084.719057764039
starting_time: 0.0


In [26]:
with NWBHDF5IO(filepath, "r", load_namespaces=True) as io:
    nwb = io.read()
    # timestamps, if explicit
    if raw.timestamps is not None:
        t = raw.timestamps[:1000]
        print(t[:10])
    else:
        # reconstruct timestamps from rate and starting_time
        import numpy as np
        t = raw.starting_time + np.arange(1000) / raw.rate
        print(t[:10])

[0.         0.0009219  0.0018438  0.00276569 0.00368759 0.00460949
 0.00553139 0.00645328 0.00737518 0.00829708]


In [27]:
with NWBHDF5IO(filepath, "r", load_namespaces=True) as io:
    nwb = io.read()
    ttl = nwb.acquisition["TTL_1"]

    print("TTL description:", ttl.description)
    print("TTL data shape:", ttl.data.shape)
    print("TTL timestamps shape:", ttl.timestamps.shape)
    print("First 20 TTL timestamps:", ttl.timestamps[:20])


TTL description: TTL 1 events from EEG annotations (SYNC_1 onsets)
TTL data shape: (583,)
TTL timestamps shape: (583,)
First 20 TTL timestamps: [ 8.7414  9.7426 10.7447 11.8454 12.8466 13.8496 14.9504 15.9516 16.9537
 18.0553 19.0565 20.0586 21.1603 22.1624 23.1645 24.2652 25.2664 26.2676
 27.3702 28.3714]
First 20 TTL values: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]


In [34]:
with NWBHDF5IO(filepath, "r", load_namespaces=True) as io:
    nwb = io.read()
    raw = nwb.acquisition["raw_EEG"]
    filt = nwb.acquisition["filtered_EEG"]

    print("Min:", np.min(raw.data[:10000, :]))
    print("Max:", np.max(raw.data[:10000, :]))
    print("Any nonzero?", np.any(raw.data[:10000, :] != 0))
    print("Global min/max on a later chunk:",
        np.min(raw.data[100000:101000, :]),
        np.max(raw.data[100000:101000, :]))
    
    print("conversion:", raw.conversion)
    print("offset:", raw.offset)
    print("resolution:", raw.resolution)
    print("unit:", raw.unit)


Min: 0.0
Max: 0.009095238095238095
Any nonzero? True
Global min/max on a later chunk: 0.0 0.008952380952380953
conversion: 1.0
offset: 0.0
resolution: -1.0
unit: volts
